In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

from dataclasses import dataclass
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
import numpy as np
from matplotlib import pyplot as plt
from pydantic_yaml import parse_yaml_file_as
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.data.wiki.dswiki import WikiDsLoader
from mllm.data.utils import RandomInputTokenizer, RandomInputTokenizerV2, TokensSubset, TokensSubsetV2, tokens_subsets_to_tensors, tokens_subsets_v2_to_tensors
from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.train.encdec_graph_bert import MaskedCiteDataset, create_masked_cite_dataloader, load_split_wiki_dataset

🚨 `emb_off` is part of GPT2Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in q:\prog\mllm\notebooks\..\mllm\model\gpt2\modeling_gpt2.py.
🚨 `emb_off` is part of GPT2LMHeadModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in q:\prog\mllm\notebooks\..\mllm\model\gpt2\modeling_gpt2.py.


In [3]:
# DATA_PATH = Path(os.path.expandvars('$HOME')) / 'data'
DATA_PATH = Path('Q:/data')
# WIKI_DS_NAME = '20200501.en'
WIKI_DS_NAME = '20220301.en'

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260429_091845-pre_encdecbert20260110193915-bertbaseuncased-d768-embEncCls-inp128-decGpt2-msl384-sepF-pallF-eer4-ewn2x4-frzencF-dsCite-trn_lr5e-05_bs30'
mixed_decoder_subdir = 'mixeddecoder-20260511_210529-pre_encdecbert20260110193915-bertbaseuncased-d768-embEncCls-inp128-decQwen2.51.5b-msl384-dtypeBf16-sepF-pallF-eer4-ewn2x6-frzencF-dsCite-trn_lr5e-05_bs20'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

cpu


## Cite inference (Wiki)

### Wikipedia dataset loading

In [4]:
# dss = load_dataset('wikipedia', WIKI_DS_NAME, beam_runner='DirectRunner', cache_dir=str(DATA_PATH))
dss = load_dataset('wikipedia', WIKI_DS_NAME, cache_dir=str(DATA_PATH))
ds: Dataset = dss['train']
n_docs = len(ds)
print(f'Wikipedia {WIKI_DS_NAME} docs: {n_docs}')

Using the latest cached version of the dataset since wikipedia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration '20220301.en' at Q:\data\wikipedia\20220301.en\2.0.0\d41137e149b2ea90eead07e7e3f805119a8c22dd1d5b61651af8e3e3ee736001 (last modified on Sun Nov 16 11:14:09 2025).


Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia 20220301.en docs: 6458670


### Masked dataset loading

In [6]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
pprint(model_cfg.dict())

tkz_enc = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
tkz_dec = AutoTokenizer.from_pretrained(model_cfg.decoder_model_name)
inp_len = model_cfg.enc_bert.inp_len
n_special_toks = 3
prompt_all = model_cfg.prompt_all
mask_cfg = None
ds_cite = MaskedCiteDataset(
    ds, tkz_enc, max_seq_len=inp_len, n_special_toks=n_special_toks,
    mask_cfg=mask_cfg, prompt_all=prompt_all, device=device, tkz_dec=tkz_dec,
)


{'d_model': 768,
 'decoder_dtype': <DecoderDtype.Bf16: 'bf16'>,
 'decoder_model_name': 'Qwen/Qwen2.5-1.5B',
 'decoder_only': False,
 'decoder_type': <MixedDecoderType.Qwen: 'qwen'>,
 'emb_exp_rate': 4,
 'emb_win_max_size': 6,
 'emb_win_min_size': 2,
 'enc_bert': {'d_model': 768,
              'emb2_tok_name': '',
              'emb_type': <BertEmbType.Cls: 'cls'>,
              'inp_len': 128,
              'pad_token_id': 0,
              'pretrained_model_name': 'bert-base-uncased',
              'tokenizer_name': 'bert-base-uncased'},
 'max_seq_len': 384,
 'min_next_toks': 64,
 'prompt_all': False,
 'train_cfg': {'batch_size': 20,
               'freeze_encoder': False,
               'learning_rate': 2.5e-05,
               'learning_rate_scheduler': {'cls_name': 'ReduceLROnPlateau',
                                           'module_path': 'torch.optim.lr_scheduler',
                                           'params': {'factor': 0.5,
                                              

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Inspect a masked dataset batch samples

In [7]:
batch_off = 10
batch_size = 5
inds = np.arange(batch_off, batch_off + batch_size)
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

Token indices sequence length is longer than the specified maximum sequence length for this model (9432 > 512). Running this sequence through the model will result in indexing errors


In [8]:
print('tkz_enc:', tkz_enc)
print('tkz_dec:', tkz_dec)


tkz_enc: BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
tkz_dec: Qwen2TokenizerFast(name_or_path='Qwen/Qwen2.5-1.5B', vocab_size=151643, model_max_length=131072, is_fast=Tru

In [9]:
print(f'inp_toks: {batch.inp_toks.shape}. inp_attn_mask: {batch.inp_att_mask.shape}. first: {batch.inp_toks[0, 0]}. last: {batch.inp_toks[0, -1]}.')
print(f'prompts_toks: {batch.prompts_toks.shape}. prompts_attn_mask: {batch.prompts_att_mask.shape}. first: {batch.prompts_toks[0, 0]}. last: {batch.prompts_toks[0, -1]}.')
print(f'cites_toks: {batch.cites_toks.shape}. cites_attn_mask: {batch.cites_att_mask.shape}.')

inp_toks: torch.Size([5, 128]). inp_attn_mask: torch.Size([5, 128]). first: 101. last: 102.
prompts_toks: torch.Size([5, 34]). prompts_attn_mask: torch.Size([5, 34]). first: 34. last: 13.
cites_toks: torch.Size([5, 70]). cites_attn_mask: torch.Size([5, 70]).


In [10]:
print('inp_toks[0]:', tkz_enc.decode(batch.inp_toks[0].cpu().numpy()))
print('prompts_toks[0]:', tkz_dec.decode(batch.prompts_toks[0].cpu().numpy()))
print('cites_toks[0]:', tkz_dec.decode(batch.cites_toks[0].cpu().numpy()))


inp_toks[0]: [CLS] the first six ceremonies, the eligibility period spanned two calendar years. at the 29th ceremony, held in 1957, the best foreign language film category, now known as best international feature film, was introduced. until then, foreign - language films had been honored with the special achievement award. perhaps the most widely seen streaker in history was 34 - year - old robert opel, who streaked across the stage of the dorothy chandler pavilion in los angeles flashing a peace sign on lille widespread rear nationalabe homo [unused287] us television at the 46th academy awards in 1974. bemused host david niven quipped, " isn ' t it fascinating [SEP]
prompts_toks[0]: Cite tag begin: "lille widespread rear". Cite tag end: "##abe homo [unused287]". Produce output text between these tags.
cites_toks[0]: national<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endof

### Model loading and inference

In [11]:
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz_enc, tkz_dec).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None


`torch_dtype` is deprecated! Use `dtype` instead!


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Load 538


### Run forward pass (with windowing disabled for inference)

In [12]:

batch_off = 10
# batch_size = 4 * model_cfg.emb_win_max_size
batch_size = 15
inds = np.arange(batch_size) + batch_off
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

In [13]:
with torch.no_grad():
    loss_dict, dec_logits = model.run_on_text_citation(batch)
print(loss_dict)

{'loss': tensor(0.7178)}


In [14]:
logits = dec_logits.detach().numpy()
logits.shape

(15, 181, 151936)

In [15]:
probs_pred = torch.softmax(dec_logits, dim=-1)
print('probs shape:', probs_pred.shape)
docs_toks_out = torch.argmax(probs_pred, dim=-1)
docs_toks_out = docs_toks_out.detach().numpy()
print('toks_out shape:', docs_toks_out.shape)

probs shape: torch.Size([15, 181, 151936])
toks_out shape: (15, 181)


In [16]:
n_ctx = model_cfg.emb_win_max_size
if model.cfg.emb_exp_rate > 0:
    n_ctx = n_ctx * model.cfg.emb_exp_rate
prefix_len = n_ctx
if model.cfg.use_sep:
    prefix_len += 1
prefix_len += batch.prompts_toks.shape[1]
target_start_idx = prefix_len - 1

if model.cfg.prompt_all:
    target_len = batch.inp_toks.shape[1] - 1  # strip CLS
else:
    target_len = batch.cites_toks.shape[1]

print(f'n_ctx: {n_ctx}, prefix_len: {prefix_len}, target_start_idx: {target_start_idx}, target_len: {target_len}')

# Extract predicted tokens for the target region
target_logits = dec_logits[:, target_start_idx:target_start_idx + target_len, :]
target_toks_pred = torch.argmax(target_logits, dim=-1).detach().numpy()
print('target_toks_pred shape:', target_toks_pred.shape)

n_ctx: 24, prefix_len: 61, target_start_idx: 60, target_len: 121
target_toks_pred shape: (15, 121)


In [17]:
for i in range(batch_size):
    print(f'=== Sample {i} ===')
    if model.cfg.prompt_all:
        gt_toks = batch.inp_toks[i, 1:]  # strip CLS
        print('ground truth (enc tkz):')
        print(tkz_enc.decode(gt_toks.cpu().numpy()))
    else:
        gt_toks = batch.cites_toks[i]
        print('ground truth (dec tkz):')
        print(tkz_dec.decode(gt_toks.cpu().numpy()))
    print('predicted (dec tkz):')
    print(tkz_dec.decode(target_toks_pred[i]))
    print()


=== Sample 0 ===
ground truth (dec tkz):
, detailing his criticisms, for which there was booing and cheering by the audience. disqualifications six films have had nominations revoked before the official award ceremony : the circus ( 1928 ) – the film was voluntarily removed by the academy from competitive categories, to award charlie chaplin a special award. hondo ( 1953 ) – removed from the best story ballot after letters from the producer and nominee questioned its inclusion in the category. high society ( 1955 ) – withdrawn from screen<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
predicted (dec tkz):
film detailing his ", and which he was booing and applause by the audience. screenaffectedifications six films had had directors before before the ceremony ceremony ceremony : the circus ( 1948 ) – the film was expelled withdrawn from the association f

### Encoder embeddings inspection

In [18]:
with torch.no_grad():
    inp_enc_embs = model.run_enc(batch.inp_toks, batch.inp_att_mask)
print('inp_enc_embs shape:', inp_enc_embs.shape)

inp_embs = inp_enc_embs.cpu().detach().numpy()
print('min:', inp_embs.min(), 'max:', inp_embs.max(), 'mean:', inp_embs.mean(), 'std:', inp_embs.std())

inp_enc_embs shape: torch.Size([15, 768])
min: -3.0394013 max: 2.363718 mean: -0.017065661 std: 0.6211807


### Autoregressive generation from encoder embeddings

In [23]:
@torch.no_grad()
def generate_from_embs(
    model: MixedDecoder, inp_enc_embs: torch.Tensor,
    prompt_toks: torch.Tensor, prompt_att_mask: torch.Tensor,
    max_new_tokens: int = 100, temperature: float = 1.0,
    win_size: Optional[int] = None,
) -> torch.Tensor:
    """Autoregressive generation: given encoder CLS embeddings and prompt tokens,
    generate tokens one by one using the MixedDecoder.

    Args:
        model: MixedDecoder model in eval mode
        inp_enc_embs: (batch_size, d_model) - CLS embeddings from encoder
        prompt_toks: (batch_size, prompt_len) - prompt token ids
        prompt_att_mask: (batch_size, prompt_len) - prompt attention mask
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (1.0 = greedy with argmax)
        win_size: size of embedding window per sample (None = use all batch embeddings)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    batch_size = inp_enc_embs.shape[0]
    device = inp_enc_embs.device
    cfg = model.cfg

    # Determine embedding window indices
    win_indices = None
    if win_size is not None and win_size < batch_size:
        offset_before = 0  # sample's own embedding at position 0
        sample_idx = torch.arange(batch_size, device=device)
        j_idx = torch.arange(win_size, device=device)
        # win_indices[i, j] = (i - offset_before + j) % batch_size
        win_indices = (sample_idx.unsqueeze(1) - offset_before + j_idx.unsqueeze(0)) % batch_size

    # Build context embeddings
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(inp_enc_embs)
        exp_embs = exp_embs.view(batch_size, cfg.emb_exp_rate, model.d_dec)
        if win_indices is not None:
            # (batch_size, win_size, emb_exp_rate, d_dec) -> (batch_size, win_size * emb_exp_rate, d_dec)
            ctx_embs = exp_embs[win_indices]
            ctx_embs = ctx_embs.reshape(batch_size, win_indices.shape[1] * cfg.emb_exp_rate, model.d_dec)
        else:
            exp_embs = exp_embs.reshape(1, batch_size * cfg.emb_exp_rate, model.d_dec)
            ctx_embs = exp_embs.expand(batch_size, -1, -1)
    else:
        if win_indices is not None:
            ctx_embs = inp_enc_embs[win_indices]
        else:
            ctx_embs = inp_enc_embs.unsqueeze(0).expand(batch_size, -1, -1)

    # Project if needed
    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]

    # Build initial prefix: [ctx_embs, (sep), prompt_embs]
    parts_embs = [ctx_embs]
    parts_mask = [torch.ones((batch_size, n_ctx), dtype=torch.long, device=device)]

    if cfg.use_sep:
        sep_tok = torch.full((batch_size, 1), model.sep_token_id, dtype=torch.long, device=device)
        sep_emb = model.word_embeddings(sep_tok)
        parts_embs.append(sep_emb)
        parts_mask.append(torch.ones((batch_size, 1), dtype=torch.long, device=device))

    prompt_embs = model.word_embeddings(prompt_toks)
    parts_embs.append(prompt_embs)
    parts_mask.append(prompt_att_mask)

    input_embs = torch.cat(parts_embs, dim=1)  # (batch_size, prefix_len, d_dec)
    attention_mask = torch.cat(parts_mask, dim=1)

    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        if model.pos_emb is not None:
            pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
            embs_with_pos = input_embs + model.pos_emb(pos_ids)
        else:
            # Qwen and other RoPE-based decoders handle positions internally.
            embs_with_pos = input_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]  # (batch_size, vocab_size)

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)  # (batch_size,)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        # Append new token embedding
        next_emb = model.word_embeddings(next_tok.unsqueeze(1))  # (batch_size, 1, d_dec)
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        # Technically this isn't correct in case prompts have different lengths, but in our setting all prompts have the same length

        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)
    return torch.stack(generated_ids, dim=1)  # (batch_size, max_new_tokens)


In [24]:
# batch_off = 10
# batch_size = 15
batch_off = 25
batch_size = 10
inds = np.arange(batch_size) + batch_off
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

inp_enc_embs = model.run_enc(batch.inp_toks, batch.inp_att_mask)

In [25]:
max_new_tokens = 100
win_size = model_cfg.emb_win_max_size
gen_toks = generate_from_embs(
    model, inp_enc_embs,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
    win_size=win_size,
)
print('gen_toks shape:', gen_toks.shape)

inp_toks = batch.inp_toks.cpu().numpy()
prompts_toks = batch.prompts_toks.cpu().numpy()
cites_toks = batch.cites_toks.cpu().numpy()
gen_toks_np = gen_toks.cpu().numpy()

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input (enc tkz):')
    print(tkz_enc.decode(inp_toks[i]))
    print('prompt (dec tkz):')
    print(tkz_dec.decode(prompts_toks[i]))
    print('ground truth (dec tkz):')
    print(tkz_dec.decode(cites_toks[i]))
    print('generated (dec tkz):')
    print(tkz_dec.decode(gen_toks_np[i]))
    print()


gen_toks shape: torch.Size([10, 100])
=== Sample 0 ===
input (enc tkz):
[CLS]i ( india ), viscii ( vietnam fuselage chart nightstand ). although these encodings are sometimes referred to as ascii, true ascii is defined strictly only by the ansi standard. most early home computer systems developed their own 8 - bit character sets containing line - drawing and game glyphs, and often filled in some or all of the control characters from 0 to 31 waitress 249te with more graphics. kaypro cp / m computers used the " upper " 128 characters for the greek alphabet. the petscii code commodore international used for their 8 - bit systems is probably unique among post - 1970 [SEP]
prompt (dec tkz):
Cite tag begin: "fuselage chart nightstand". Cite tag end: "waitress 249te". Produce output text between these tags.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
ground truth (dec tkz):
). although these encodings are sometimes referred to as ascii, true asci

### Decoding with noisy embeddings

In [ ]:
# Add small noise to encoder embeddings and compare generation quality
noise_std = 1e-2
noise = torch.randn_like(inp_enc_embs) * noise_std
inp_enc_embs_noisy = inp_enc_embs + noise

gen_toks_noisy = generate_from_embs(
    model, inp_enc_embs_noisy,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
    win_size=win_size,
)

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input (enc tkz):')
    print(tkz_enc.decode(batch.inp_toks[i].cpu().numpy()))
    print('generated clean (dec tkz):')
    print(tkz_dec.decode(gen_toks[i].cpu().numpy()))
    print('generated noisy std={:.0e} (dec tkz):'.format(noise_std))
    print(tkz_dec.decode(gen_toks_noisy[i].cpu().numpy()))
    print()


=== Sample 0 ===
input:
[CLS] olympic games. another courtesy melted lone reason for the move to late february and early march is also to avoid the awards ceremony occurring so close to the religious holidays of passover and easter, which for decades had been a grievance from members and the general public. advertising is somewhat restricted, however, as traditionally no movie studios or competitors of official academy award sponsors may advertise during the telecast. the production of the academy awards telecast currently holds the distinction of winning the most emmys in history, with 47 wins and 195 nominations overall since that award magnetic compliant emanuel ' s own launch in 1949. after many years of being held [SEP]
generated (clean):
reason for their move to be late in the fifth and last months. the international symposium will be frequently prepared for views of the daytime to avoid straight and a compromise of members and fblies in the year. history the conference is diffic

### GPT2 tokenizer test

In [ ]:
gpt2_tkz = AutoTokenizer.from_pretrained('gpt2')

In [ ]:
for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz_enc.decode(inp_toks[i]))
    print(gpt2_tkz.decode(inp_toks[i]))
    print('prompt:')
    print(tkz_dec.decode(prompts_toks[i]))
    print(gpt2_tkz.decode(prompts_toks[i]))
    print('ground truth:')
    print(tkz_dec.decode(cites_toks[i]))
    print(gpt2_tkz.decode(cites_toks[i]))
    print('generated:')
    print(tkz_dec.decode(gen_toks_np[i]))
    print(gpt2_tkz.decode(gen_toks_np[i]))
    print()
    break


=== Sample 0 ===
input:
[CLS] in 2015. originally based on letterman lord arrows the english alphabet, ascii encodes 128 specified characters into seven - bit integers as shown by the ascii chart above. ninety - five of the encoded characters are printable : these include the digits 0 to 9, lowercase letters aphonic postgraduatedication to z, uppercase letters a to z, and punctuation symbols. in addition, the original ascii specification included 33 non - printing control codes which originated with teletype machines ; most of these are now obsolete, although a few are still commonly used, such as the carriage return, line feed, [SEP]
�attle upd Cl problems himself tot burger Des whisputOT Celltersged Safitorrd164 Marg glad implement fire laun takehel hereinged star Myputged SafitorNot34 Cl harmful take recent anythingput..... implementators begins staff Deolution requireput fortunway mindiststers Def administrator drink helplys reminder detainees mind detters movement administrator dr

## QnA inference (SQuAD v2)

### Model loading and inference

In [ ]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
tkz_enc = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
tkz_dec = AutoTokenizer.from_pretrained(model_cfg.decoder_model_name)
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size
inp_len = model_cfg.enc_bert.inp_len

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz_enc, tkz_dec).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None


Load 351


### Load SQuAD v2 dataset and create QnA batch

In [ ]:
from mllm.train.qna_cite import QnaCiteBatch, QnaCiteDataset, load_split_squadv2

df_sq, sq_inds_train, sq_inds_val = load_split_squadv2(exclude_empty_answers=True)
max_chunks = max(model_cfg.emb_win_max_size, 1)
ds_qna = QnaCiteDataset(
    df_sq, sq_inds_val, tkz_enc, inp_len=inp_len, max_chunks=max_chunks,
    max_ans_toks=100, max_prompt_toks=100, device=device, tkz_dec=tkz_dec,
)
print(f'QnA dataset size: {len(ds_qna)}. max_chunks: {max_chunks}')


Remove empty answers from dataset squad_v2. Size: 142192 --> 92749
R0. SQuAD v2 n_total=92749. n_train=88112. n_val=4637.
QnA dataset size: 4637. max_chunks: 10


In [38]:
qna_batch_off = 0
qna_batch_size = 5
# sq_inds = sq_inds_train
sq_inds = sq_inds_val
qna_inds = sq_inds[qna_batch_off:qna_batch_off + qna_batch_size]
qna_batch = ds_qna.get_batch(qna_inds)
print(f'ctx_chunks_toks: {qna_batch.ctx_chunks_toks.shape}')
print(f'ctx_chunk_counts: {qna_batch.ctx_chunk_counts}')
print(f'prompt_toks: {qna_batch.prompt_toks.shape}')
print(f'prompt_lengths: {qna_batch.prompt_lengths}')
print(f'ans_toks: {qna_batch.ans_toks.shape}')
print(f'ans_att_mask sum per sample: {qna_batch.ans_att_mask.sum(dim=1).tolist()}')

ctx_chunks_toks: torch.Size([7, 128])
ctx_chunk_counts: [1, 2, 1, 2, 1]
prompt_toks: torch.Size([5, 19])
prompt_lengths: [19, 16, 12, 18, 18]
ans_toks: torch.Size([5, 10])
ans_att_mask sum per sample: [3, 5, 5, 4, 10]


### Inspect QnA batch tokens

In [ ]:
chunk_offset = 0
for i in range(qna_batch_size):
    n_chunks = qna_batch.ctx_chunk_counts[i]
    print(f'=== Sample {i} ({n_chunks} context chunks) ===')
    for ci in range(n_chunks):
        chunk_toks = qna_batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
        print(f'  chunk {ci}: {tkz_enc.decode(chunk_toks, skip_special_tokens=False)}')
    chunk_offset += n_chunks

    prompt_len = qna_batch.prompt_lengths[i]
    prompt_tok_ids = qna_batch.prompt_toks[i, :prompt_len].cpu().numpy()
    print(f'  prompt ({prompt_len} toks): {tkz_dec.decode(prompt_tok_ids)}')

    ans_len = int(qna_batch.ans_att_mask[i].sum().item())
    ans_tok_ids = qna_batch.ans_toks[i, :ans_len].cpu().numpy()
    print(f'  answer ({ans_len} toks): {tkz_dec.decode(ans_tok_ids)}')

    print(f'  prompt token ids: {prompt_tok_ids.tolist()}')
    print(f'  answer token ids: {ans_tok_ids.tolist()}')
    print()


=== Sample 0 (1 context chunks) ===
  chunk 0: [CLS] the precepts are not formulated as imperatives, but as training rules that laypeople undertake voluntarily to facilitate practice. in buddhist thought, the cultivation of dana and ethical conduct themselves refine consciousness to such a level that rebirth in one of the lower heavens is likely, even if there is no further buddhist practice. there is nothing improper or un - buddhist about limiting one ' s aims to this level of attainment. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
  prompt (19 toks): question : even if there is no further buddhist practice, what heavens is still likely? answer :
  answer (3 toks): [CLS] lower [SEP]
  prompt token ids: [3160, 1024, 2130, 2065, 2045, 2003, 2053, 2582, 7992, 3218, 1010, 2054, 17223

### QnA forward pass (teacher-forced)

In [40]:
with torch.no_grad():
    qna_loss_dict, qna_logits = model.run_on_qna(qna_batch)
print('QnA loss:', qna_loss_dict)
print('QnA logits shape:', qna_logits.shape)

QnA loss: {'loss': tensor(2.5932)}
QnA logits shape: torch.Size([5, 68, 50257])


In [ ]:
# Decode teacher-forced predictions for the answer region
cfg = model.cfg
emb_win_max = cfg.emb_win_max_size
n_ctx_qna = emb_win_max
if cfg.emb_exp_rate > 0:
    n_ctx_qna = n_ctx_qna * cfg.emb_exp_rate
sep_len = 1 if cfg.use_sep else 0

for i in range(qna_batch_size):
    pl = qna_batch.prompt_lengths[i]
    al = int(qna_batch.ans_att_mask[i].sum().item())
    target_start = n_ctx_qna + sep_len + pl - 1  # last prompt position predicts first answer token

    pred_logits = qna_logits[i, target_start:target_start + al, :]
    pred_toks = torch.argmax(pred_logits, dim=-1).cpu().numpy()

    gt_toks = qna_batch.ans_toks[i, :al].cpu().numpy()

    print(f'=== Sample {i} ===')
    print(f'  prompt:     {tkz_dec.decode(qna_batch.prompt_toks[i, :pl].cpu().numpy())}')
    print(f'  GT answer:  {tkz_dec.decode(gt_toks)}')
    print(f'  predicted:  {tkz_dec.decode(pred_toks)}')
    print(f'  GT tok ids: {gt_toks.tolist()}')
    print(f'  pred tok ids: {pred_toks.tolist()}')
    print()


=== Sample 0 ===
  prompt:     question : even if there is no further buddhist practice, what heavens is still likely? answer :
  GT answer:  [CLS] lower [SEP]
  predicted:  [CLS] middle part
  GT tok ids: [101, 2896, 102]
  pred tok ids: [101, 2690, 2112]

=== Sample 1 ===
  prompt:     question : how many italians lived in libya prior to october of 1970? answer :
  GT answer:  [CLS] 12, 000 [SEP]
  predicted:  [CLS] 250, 000 [SEP]
  GT tok ids: [101, 2260, 1010, 2199, 102]
  pred tok ids: [101, 5539, 1010, 2199, 102]

=== Sample 2 ===
  prompt:     question : when was the genpei war? answer :
  GT answer:  [CLS] late 12th century [SEP]
  predicted:  [CLS] april 1980s century [SEP]
  GT tok ids: [101, 2397, 5940, 2301, 102]
  pred tok ids: [101, 2258, 3865, 2301, 102]

=== Sample 3 ===
  prompt:     question : how long did the us and its coalition partners have to occupy iraq? answer :
  GT answer:  [CLS] 9 years [SEP]
  predicted:  [CLS] 250 years [SEP]
  GT tok ids: [101, 1023, 2086

### Autoregressive QnA generation

In [ ]:
@torch.no_grad()
def generate_qna(
    model: MixedDecoder, qna_batch: QnaCiteBatch,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation for QnA: encode context chunks, build per-sample
    context embedding windows, then generate answer tokens one by one.

    Args:
        model: MixedDecoder model in eval mode
        qna_batch: QnaCiteBatch with context chunks, prompts, and answer tokens
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (<=0 means greedy argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    cfg = model.cfg
    batch_size = len(qna_batch.ctx_chunk_counts)
    device = qna_batch.ctx_chunks_toks.device

    # 1. Encode all context chunks
    all_enc_embs = model.run_enc(qna_batch.ctx_chunks_toks, qna_batch.ctx_chunks_att_mask)
    total_chunks = all_enc_embs.shape[0]

    # 2. Build per-sample context windows (own chunks + zero-padding if needed)
    emb_win_max = max(cfg.emb_win_max_size, 1)
    target_win_size = emb_win_max

    chunk_offsets = [0]
    for c in qna_batch.ctx_chunk_counts:
        chunk_offsets.append(chunk_offsets[-1] + c)

    ctx_embs_raw = torch.zeros((batch_size, target_win_size, cfg.d_model), device=device)
    for i in range(batch_size):
        start, end = chunk_offsets[i], chunk_offsets[i + 1]
        own = all_enc_embs[start:end]
        n_own = min(own.shape[0], target_win_size)
        ctx_embs_raw[i, :n_own] = own[:n_own]
        # Remaining slots stay zero (no random fillers during inference)

    # 3. Apply emb_exp expansion or projection
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(ctx_embs_raw)
        exp_embs = exp_embs.view(batch_size, target_win_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs
    else:
        ctx_embs = ctx_embs_raw

    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]
    sep_len = 1 if cfg.use_sep else 0

    # 4. Build per-sample initial prefix: [ctx_embs, (sep), prompt_embs]
    prompt_lens = qna_batch.prompt_lengths
    max_prefix_len = n_ctx + sep_len + max(prompt_lens)

    input_embs = torch.zeros((batch_size, max_prefix_len, model.d_dec), device=device)
    attention_mask = torch.zeros((batch_size, max_prefix_len), dtype=torch.long, device=device)

    prompt_embs_all = model.word_embeddings(qna_batch.prompt_toks)
    if cfg.use_sep:
        sep_emb = model.word_embeddings(
            torch.full((1, 1), model.sep_token_id, dtype=torch.long, device=device)
        )

    actual_prefix_lens = []
    for i in range(batch_size):
        pos = 0
        # Context embeddings
        input_embs[i, :n_ctx] = ctx_embs[i]
        attention_mask[i, :n_ctx] = 1
        pos = n_ctx

        # Optional SEP
        if cfg.use_sep:
            input_embs[i, pos:pos + 1] = sep_emb[0]
            attention_mask[i, pos:pos + 1] = 1
            pos += 1

        # Prompt tokens (actual length)
        pl = prompt_lens[i]
        input_embs[i, pos:pos + pl] = prompt_embs_all[i, :pl]
        attention_mask[i, pos:pos + pl] = 1
        pos += pl
        actual_prefix_lens.append(pos)

    # 5. Autoregressive generation
    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        if model.pos_emb is not None:
            pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
            embs_with_pos = input_embs + model.pos_emb(pos_ids)
        else:
            # Qwen and other RoPE-based decoders handle positions internally.
            embs_with_pos = input_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]  # (batch_size, vocab_size)

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        next_emb = model.word_embeddings(next_tok.unsqueeze(1))
        input_embs = torch.cat([input_embs, next_emb], dim=1)

        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)    return torch.stack(generated_ids, dim=1)


In [45]:
# qna_gen_batch_off = 0
# qna_gen_batch_size = 5
# qna_gen_inds = list(range(qna_gen_batch_off, qna_gen_batch_off + qna_gen_batch_size))
# qna_gen_batch = ds_qna.get_batch(qna_gen_inds)
qna_gen_batch = qna_batch

qna_max_new_tokens = 60
qna_gen_toks = generate_qna(
    model, qna_gen_batch,
    max_new_tokens=qna_max_new_tokens, temperature=0,
)
print('qna_gen_toks shape:', qna_gen_toks.shape)

qna_gen_toks shape: torch.Size([5, 60])


In [ ]:
# Display QnA generation results with token inspection
qna_gen_np = qna_gen_toks.cpu().numpy()
qna_chunk_offset = 0
for i in range(qna_gen_batch_size):
    n_chunks = qna_gen_batch.ctx_chunk_counts[i]
    pl = qna_gen_batch.prompt_lengths[i]
    al = int(qna_gen_batch.ans_att_mask[i].sum().item())
    gt_ans_ids = qna_gen_batch.ans_toks[i, :al].cpu().numpy()
    prompt_ids = qna_gen_batch.prompt_toks[i, :pl].cpu().numpy()
    gen_ids = qna_gen_np[i]

    print(f'=== Sample {i} ({n_chunks} context chunks) ===')

    # Show context chunks
    for ci in range(n_chunks):
        chunk_toks = qna_gen_batch.ctx_chunks_toks[qna_chunk_offset + ci].cpu().numpy()
        print(f'  ctx chunk {ci}: {tkz_enc.decode(chunk_toks, skip_special_tokens=True)[:150]}...')
    qna_chunk_offset += n_chunks

    print(f'  prompt:        {tkz_dec.decode(prompt_ids)}')
    print(f'  GT answer:     {tkz_dec.decode(gt_ans_ids)}')
    print(f'  generated:     {tkz_dec.decode(gen_ids)}')
    print(f'  prompt ids:    {prompt_ids.tolist()}')
    print(f'  GT answer ids: {gt_ans_ids.tolist()}')
    print(f'  gen ids:       {gen_ids.tolist()}')
    print()


=== Sample 0 (1 context chunks) ===
  ctx chunk 0: the precepts are not formulated as imperatives, but as training rules that laypeople undertake voluntarily to facilitate practice. in buddhist thought...
  prompt:        question : even if there is no further buddhist practice, what heavens is still likely? answer :
  GT answer:     [CLS] lower [SEP]
  generated:     [CLS] the number of buddhist teaching [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP]
  prompt ids:    [3160, 1024, 2130, 2065, 2045, 2003, 2053, 2582, 7992, 3218, 1010, 2054, 17223, 2003, 2145, 3497, 1029, 3437, 1024]
  GT answer ids: [101, 2896, 102]
  gen ids:       [101, 1996, 2193, 1997, 7992, 4252, 102, 102, 102, 102, 102, 102, 102, 102, 102, 10

## Next token inference (Wiki)

### Wiki dataset loading

In [ ]:
from mllm.train.next_tok_wiki import NextTokBatch, NextTokWikiDataset
from mllm.data.wiki.itwiki import get_split_wiki_ds

ds_wiki, wiki_inds_train, wiki_inds_val = get_split_wiki_ds(DATA_PATH, val_ratio=0.05, rand_seed=100)
print(f'Wiki docs: {len(ds_wiki)}. Train: {len(wiki_inds_train)}. Val: {len(wiki_inds_val)}.')

In [ ]:
### Config and model loading

In [ ]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
tkz_enc = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
tkz_dec = AutoTokenizer.from_pretrained(model_cfg.decoder_model_name)
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size
inp_len = model_cfg.enc_bert.inp_len

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz_enc, tkz_dec).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None


### Create NextTokWikiDataset and inspect batch

In [ ]:
min_next_toks = 20
max_target_toks = 128
emb_win_min = max(model_cfg.emb_win_min_size, 1)
emb_win_max = max(model_cfg.emb_win_max_size, 1)

ds_next = NextTokWikiDataset(
    ds_wiki, wiki_inds_val, tkz_enc, inp_len=inp_len, min_next_toks=min_next_toks,
    emb_win_min_size=emb_win_min, emb_win_max_size=emb_win_max,
    max_target_toks=max_target_toks, device=device, tkz_dec=tkz_dec,
)
print(f'NextTokWikiDataset size: {len(ds_next)}. emb_win: [{emb_win_min}, {emb_win_max}]. max_target_toks: {max_target_toks}')

next_batch_size = 5
next_batch = ds_next.get_batch(next_batch_size)
print(f'ctx_chunks_toks: {next_batch.ctx_chunks_toks.shape}')
print(f'ctx_chunk_counts: {next_batch.ctx_chunk_counts}')
print(f'prompt_toks: {next_batch.prompt_toks.shape}')
print(f'prompt_lengths: {next_batch.prompt_lengths}')
print(f'target_toks: {next_batch.target_toks.shape}')
print(f'target_att_mask sum per sample: {next_batch.target_att_mask.sum(dim=1).tolist()}')


In [ ]:
chunk_offset = 0
for i in range(next_batch_size):
    n_chunks = next_batch.ctx_chunk_counts[i]
    print(f'=== Sample {i} ({n_chunks} context chunks) ===')
    for ci in range(n_chunks):
        chunk_toks = next_batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
        print(f'  chunk {ci}: {tkz_enc.decode(chunk_toks, skip_special_tokens=True)[:200]}...')
    chunk_offset += n_chunks

    prompt_len = next_batch.prompt_lengths[i]
    prompt_tok_ids = next_batch.prompt_toks[i, :prompt_len].cpu().numpy()
    print(f'  prompt ({prompt_len} toks): {tkz_dec.decode(prompt_tok_ids)}')

    target_len = int(next_batch.target_att_mask[i].sum().item())
    target_tok_ids = next_batch.target_toks[i, :target_len].cpu().numpy()
    print(f'  target ({target_len} toks): {tkz_dec.decode(target_tok_ids)}')
    print()


### Forward pass (teacher-forced)

In [ ]:
with torch.no_grad():
    next_loss_dict, next_logits = model.run_on_next(next_batch)
print('Next-tok loss:', next_loss_dict)
print('Next-tok logits shape:', next_logits.shape)

In [ ]:
# Decode teacher-forced predictions for the target region
cfg = model.cfg
emb_win_max = max(cfg.emb_win_max_size, 1)
n_ctx_next = emb_win_max
if cfg.emb_exp_rate > 0:
    n_ctx_next = n_ctx_next * cfg.emb_exp_rate
sep_len = 1 if cfg.use_sep else 0

for i in range(next_batch_size):
    pl = next_batch.prompt_lengths[i]
    tl = int(next_batch.target_att_mask[i].sum().item())
    target_start = n_ctx_next + sep_len + pl - 1  # last prompt pos predicts first target token

    pred_logits = next_logits[i, target_start:target_start + tl, :]
    pred_toks = torch.argmax(pred_logits, dim=-1).cpu().numpy()

    gt_toks = next_batch.target_toks[i, :tl].cpu().numpy()

    print(f'=== Sample {i} ===')
    print(f'  prompt:     {tkz_dec.decode(next_batch.prompt_toks[i, :pl].cpu().numpy())}')
    print(f'  GT target:  {tkz_dec.decode(gt_toks)}')
    print(f'  predicted:  {tkz_dec.decode(pred_toks)}')
    print()


### Autoregressive next-token generation

In [ ]:
@torch.no_grad()
def generate_next_tok(
    model: MixedDecoder, next_batch: NextTokBatch,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation for next-token prediction: encode context chunks,
    build per-sample context embedding windows, then generate tokens one by one.

    Args:
        model: MixedDecoder model in eval mode
        next_batch: NextTokBatch with context chunks, prompts, and target tokens
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (<=0 means greedy argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    cfg = model.cfg
    batch_size = len(next_batch.ctx_chunk_counts)
    device = next_batch.ctx_chunks_toks.device

    # 1. Encode all context chunks
    all_enc_embs = model.run_enc(next_batch.ctx_chunks_toks, next_batch.ctx_chunks_att_mask)

    # 2. Build per-sample context windows
    emb_win_max = max(cfg.emb_win_max_size, 1)
    target_win_size = max(next_batch.ctx_chunk_counts)

    chunk_offsets = [0]
    for c in next_batch.ctx_chunk_counts:
        chunk_offsets.append(chunk_offsets[-1] + c)

    ctx_embs_raw = torch.zeros((batch_size, target_win_size, cfg.d_model), device=device)
    for i in range(batch_size):
        start, end = chunk_offsets[i], chunk_offsets[i + 1]
        n_own = end - start
        ctx_embs_raw[i, :n_own] = all_enc_embs[start:end]

    # 3. Apply emb_exp expansion or projection
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(ctx_embs_raw)
        exp_embs = exp_embs.view(batch_size, target_win_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs
    else:
        ctx_embs = ctx_embs_raw

    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]
    sep_len = 1 if cfg.use_sep else 0

    # 4. Build per-sample initial prefix: [ctx_embs, (sep), prompt_embs]
    prompt_lens = next_batch.prompt_lengths
    max_prefix_len = n_ctx + sep_len + max(prompt_lens)

    input_embs = torch.zeros((batch_size, max_prefix_len, model.d_dec), device=device)
    attention_mask = torch.zeros((batch_size, max_prefix_len), dtype=torch.long, device=device)

    prompt_embs_all = model.word_embeddings(next_batch.prompt_toks)
    if cfg.use_sep:
        sep_emb = model.word_embeddings(
            torch.full((1, 1), model.sep_token_id, dtype=torch.long, device=device)
        )

    for i in range(batch_size):
        pos = 0
        input_embs[i, :n_ctx] = ctx_embs[i]
        attention_mask[i, :n_ctx] = 1
        pos = n_ctx

        if cfg.use_sep:
            input_embs[i, pos:pos + 1] = sep_emb[0]
            attention_mask[i, pos:pos + 1] = 1
            pos += 1

        pl = prompt_lens[i]
        input_embs[i, pos:pos + pl] = prompt_embs_all[i, :pl]
        attention_mask[i, pos:pos + pl] = 1

    # 5. Autoregressive generation
    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        if model.pos_emb is not None:
            pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
            embs_with_pos = input_embs + model.pos_emb(pos_ids)
        else:
            # Qwen and other RoPE-based decoders handle positions internally.
            embs_with_pos = input_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        next_emb = model.word_embeddings(next_tok.unsqueeze(1))
        input_embs = torch.cat([input_embs, next_emb], dim=1)

        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)    return torch.stack(generated_ids, dim=1)


In [ ]:
next_max_new_tokens = 80
next_gen_toks = generate_next_tok(
    model, next_batch,
    max_new_tokens=next_max_new_tokens, temperature=0,
)
print('next_gen_toks shape:', next_gen_toks.shape)

In [ ]:
# Display next-token generation results
next_gen_np = next_gen_toks.cpu().numpy()
chunk_offset = 0
for i in range(next_batch_size):
    n_chunks = next_batch.ctx_chunk_counts[i]
    pl = next_batch.prompt_lengths[i]
    tl = int(next_batch.target_att_mask[i].sum().item())
    gt_target_ids = next_batch.target_toks[i, :tl].cpu().numpy()
    prompt_ids = next_batch.prompt_toks[i, :pl].cpu().numpy()
    gen_ids = next_gen_np[i]

    print(f'=== Sample {i} ({n_chunks} context chunks) ===')

    for ci in range(n_chunks):
        chunk_toks = next_batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
        print(f'  ctx chunk {ci}: {tkz_enc.decode(chunk_toks, skip_special_tokens=True)[:200]}...')
    chunk_offset += n_chunks

    print(f'  prompt:      {tkz_dec.decode(prompt_ids)}')
    print(f'  GT target:   {tkz_dec.decode(gt_target_ids)}')
    print(f'  generated:   {tkz_dec.decode(gen_ids)}')
    print(f'  GT tok ids:  {gt_target_ids.tolist()}')
    print(f'  gen tok ids: {gen_ids.tolist()}')
    print()
